In [1]:
pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-ollama chromadb sentence-transformers pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.3/554.3 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/2

In [3]:
!ollama pull llama3

/bin/bash: line 1: ollama: command not found


The `ollama` command is not found because the Ollama server is not installed in the Colab environment. We need to install it first.

In [7]:
# Install zstd, a dependency for Ollama
!apt-get update && apt-get install -y zstd

# Install Ollama server
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,303 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy/main 

Ollama server is installed. Now, we need to start it and then pull the `llama3` model. We will use the full path to the `ollama` executable to ensure it's found.

In [10]:
# Start Ollama server in the background using its full path
import time
!nohup /usr/local/bin/ollama serve &
# Give the server a few seconds to initialize
time.sleep(10)

nohup: appending output to 'nohup.out'


In [9]:
# Pull the llama3 model using its full path
!/usr/local/bin/ollama pull llama3

Error: could not connect to ollama server, run 'ollama serve' to start it


# Step 1: Document Loading & Splitting

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
# Please upload your PDF file to the Colab environment, e.g., to /content/
# For example, if your file is named 'apple-privacy-policy-en-ww.pdf' and you upload it to the root of your Colab session:
loader = PyPDFLoader("/content/apple-privacy-policy-en-ww.pdf")
documents = loader.load()

# Split into chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print(f"Split document into {len(chunks)} chunks.")

Split document into 37 chunks.


# Step 2:Creating the Vector Database

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Vector DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector database created successfully!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created successfully!


# Step 3: Initialize the LLM (Ollama)

In [16]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3")

# Step 4:Creating the RAG Pipeline

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Prompt
prompt = ChatPromptTemplate.from_template("""
You are a helpful assistant. Answer the question ONLY using the provided context.

<context>
{context}
</context>

Question: {question}
""")

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Format retrieved docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL Chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

The Strict Prompt: We clearly tell the LLM, “Answer the question ONLY using the provided context.” This helps prevent hallucination. We want the LLM to summarize our data, not make things up.
The Retriever (k=3): When you ask a question, the vector database finds the top three most relevant chunks.
The Chain (LCEL): LangChain Expression Language (the | symbols) moves data from one step to the next. The user’s question goes in, the retriever grabs the context, both are added to the prompt, and then Llama 3 gets the final prompt.

# Step 5: Ask the Question

In [20]:
import time
import subprocess

question = "According to the document, why user's personal data is used by Apple?"

# Ensure Ollama server is running and the llama3 model is pulled
# This logic is ideally placed earlier in the notebook, but included here
# to address the error within the constraints of this cell.

ollama_path = "/usr/local/bin/ollama"
model_name = "llama3"

# 1. Ensure Ollama server is started in the background
print(f"Ensuring Ollama server is running...")
subprocess.run(["nohup", ollama_path, "serve", "&"], check=False)
time.sleep(5) # Give it a moment to start

# 2. Wait for Ollama server to be responsive and pull model if not present
server_ready = False
for i in range(10): # Try for up to 100 seconds
    try:
        # Check if `ollama list` works, indicating server is up
        result = subprocess.run([ollama_path, "list"], capture_output=True, text=True, check=True, timeout=10)
        server_ready = True
        if model_name in result.stdout:
            print(f"Ollama server is responsive and '{model_name}' model is available.")
        else:
            print(f"Ollama server is responsive but '{model_name}' model not found. Attempting to pull...")
            pull_result = subprocess.run([ollama_path, "pull", model_name], capture_output=True, text=True, check=True, timeout=300) # Increased timeout for pull
            print(pull_result.stdout)
            print(f"'{model_name}' model pulled successfully.")

        break # Server is ready and model checked/pulled
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as e:
        print(f"Attempt {i+1}/10: Ollama server not ready or pull failed. Retrying in 10 seconds. Error: {e}")
        time.sleep(10)
    except Exception as e:
        print(f"Attempt {i+1}/10: An unexpected error occurred: {e}. Retrying in 10 seconds.")
        time.sleep(10)

if not server_ready:
    print("\n--- Fatal Error ---")
    print("Ollama server did not become responsive. Please ensure Ollama is correctly installed and started.")
else:
    # Proceed with the RAG chain invocation
    response = rag_chain.invoke(question)

    print("\n--- Answer ---")
    print(response)

Ensuring Ollama server is running...
Ollama server is responsive but 'llama3' model not found. Attempting to pull...

'llama3' model pulled successfully.

--- Answer ---
According to the document, Apple uses personal data for the following purposes:

1. To power its services.
2. To process transactions.
3. To communicate with users.
4. For security and fraud prevention.
5. To comply with law.

Additionally, Apple may use personal data for other purposes with user consent.


That’s the process for building a document Q&A system with Vector Databases and RAG.